# StarGal Full Run

Fully detailed run to compute photometric redshift of a,mixed catalog (star+galaxy)

In [ ]:
### Libs ###
from imports import *

base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(base_dir)

# Custom libraries
from scripts import statsplot as ssp
from scripts import utils as sutils

%matplotlib inline

In [ ]:
import os
from importlib import reload
import lephare as lp; reload(lp)

In [ ]:
config = lp.default_stargal_config.copy()

config.update({"VERBOSE": "NO", "Z_STEP": "0.01,0.,2.",})

config = lp.default_stargal_config.copy()

## Prepare MagGal libraries

### DC1 - Make filter library

In [ ]:
##### Create filter library #####
filter_path = os.path.join(base_dir, 'simulated_sed/buzzard_base/filt') 
print(filter_path)
filter_rep = lp.keyword("FILTER_REP", filter_path)
filter_list = lp.keyword("FILTER_LIST",
"BuzzardLSSTu.res,BuzzardLSSTg.res,BuzzardLSSTr.res,\
     BuzzardLSSTi.res,BuzzardLSSTz.res,BuzzardLSSTy4.res")

filterLib = lp.Filter(config_keymap={"FILTER_REP":filter_rep,
                                   "FILTER_LIST":filter_list,
                                   "TRANS_TYPE":lp.keyword("TRANS_TYPE", "0"),
                                   "FILTER_CALIB":lp.keyword("FILTER_CALIB", "0"), #"0,0,0,0,0,0"
                                   "FILTER_FILE":lp.keyword("FILTER_FILE", "photozdc1")})

filterLib.run()

### DC1- Make SED library

In [ ]:
##### Create SED library #####

SED_path = os.path.join(base_dir, 'simulated_sed/buzzard_base/lib_bin/updated_Buzzard_SEDs')

# SED_list = sorted([
#     f for f in os.listdir(SED_path)
#     if os.path.isfile(os.path.join(SED_path, f)) and f.endswith(".sed")
# ])

SED_list = [f'kmeansbuzzard_{i}.sed' for i in range(0,100)]

#save to file.list because we must do it like this
SED_list_path = os.path.join(SED_path, "updated_Buzzard_SEDs.list")
with open(SED_list_path, "w") as f:
    for sed_file in SED_list:
        f.write("Buzzard/"+sed_file + "\n")

SED_rep = lp.keyword("SED_REP", SED_path)
print(SED_rep)

###add SED to cahe file because ...###
import shutil

#Lephare cache file
LEPHARE_SED_GAL_PATH = os.path.expanduser("~/.cache/lephare/data/sed/GAL/Buzzard")

#Create cache file in case
os.makedirs(LEPHARE_SED_GAL_PATH, exist_ok=True)

# Cpypast all file to cache
for sed_file in SED_list + ["updated_Buzzard_SEDs.list"]:
    src = os.path.join(SED_path, sed_file)
    dst = os.path.join(LEPHARE_SED_GAL_PATH, sed_file)
    shutil.copyfile(src, dst)

print(f"past {len(SED_list)} file.sed and file.list in {LEPHARE_SED_GAL_PATH}")



SED_list_path = f"{base_dir}/simulated_sed/buzzard_base/lib_bin/updated_Buzzard_SEDs/updated_Buzzard_SEDs.list"
sedLib = lp.Sedtolib(config_keymap={
    "SED_REP": SED_rep,
    "GAL_SED": lp.keyword("GAL_SED", SED_list_path),
    "GAL_FSCALE": lp.keyword("GAL_FSCALE", "1.0"),
    "GAL_LIB": lp.keyword("GAL_LIB", 'buzzard_SEDs')},)
sedLib.run(typ="G")

### STARS - Make filter library

In [ ]:
#--- From DES filters ---
filter_path = os.path.join(base_dir, 'catalogs/DES/DES_STARCAT/WORK_COMPLETE2/filt') #paste your relative filter path
print(filter_path)
filter_rep = lp.keyword("FILTER_REP", filter_path)
filter_list = lp.keyword("FILTER_LIST",
                         "DES_filter_g.res,DES_filter_r.res,\
                         DES_filter_i.res,DES_filter_z.res,DES_filter_Y.res") #edit your filter list
#crash if add comma

filterLib2 = lp.Filter(config_keymap={"FILTER_REP":filter_rep,
                                   "FILTER_LIST":filter_list,
                                   "TRANS_TYPE":lp.keyword("TRANS_TYPE", "0"),
                                   "FILTER_CALIB":lp.keyword("FILTER_CALIB", "0"),
                                   "FILTER_FILE":lp.keyword("FILTER_FILE", "photozDES")})


filterLib2.run()

### STARS - Make SED library

In [ ]:
#--- With Pickles ---
SED_list_path = f"{base_dir}/catalogs/DES/DES_STARCAT/WORK_COMPLETE2/lib_bin/DES_STAR.list"
sedLib = lp.Sedtolib(config_keymap={
    "STAR_SED": lp.keyword("STAR_SED", SED_list_path),
    "STAR_FSCALE": lp.keyword("STAR_FSCALE", "3.0e-9"),
    "STAR_LIB": lp.keyword("STAR_LIB", 'PICKLES_SEDs')},)
sedLib.run(typ="S")

Alternative: star sed grid as a function of Teff, log(g) and Fe/H, from the BT library

In [ ]:
#--- With BT ---
SED_list_path = f"{base_dir}/simulated_sed/bt_spectra/bt_star_sed_full.list"
sedLib = lp.Sedtolib(config_keymap={
    "STAR_SED": lp.keyword("STAR_SED", SED_list_path),
    "STAR_FSCALE": lp.keyword("STAR_FSCALE", "3.0e-9"),
    "STAR_LIB": lp.keyword("STAR_LIB", 'BT_SEDs')},)
sedLib.run(typ="S")

## Run MagGal

Create a magnitude library from the SED and filter libraries.
To compare properly our available libraries, we need to run 4 times mag_gal. 

In [ ]:
maglib = lp.MagGal(config_keymap=lp.all_types_to_keymap(config))

### buzzard in LSST filters (for DC1, only gal)

In [ ]:
maglib.run(typ="G",
    gal_lib_in="buzzard_SEDs",
    gal_lib_out="buzzard_LSST",
    filter_file="photozdc1",
    magtype="AB",
    mod_extinc="0",
    extinc_law="calzetti.dat",
    em_dispersion="0",
    lib_ascii="YES",
    z_step="0.01,0.,2.")


### buzzard in DES filters (for DESstars, star-gal)

In [ ]:
maglib.run(typ="G",
    gal_lib_in="buzzard_SEDs",
    gal_lib_out="buzzard_DES",
    filter_file="photozDES",
    magtype="AB",
    mod_extinc="0",
    extinc_law="calzetti.dat",
    em_dispersion="0",
    lib_ascii="YES",
    z_step="0.01,0.,2.")

### PICKLES in DES filters (for DESstars, only star)

In [ ]:
maglib.run(typ="S",
    star_lib_in="PICKLES_SEDs",
    star_lib_out="PICKLES_DES",
    filter_file="photozDES",
    magtype="AB",
    mod_extinc="0",
    extinc_law="calzetti.dat",
    em_dispersion="0",
    lib_ascii="YES",
    z_step="0.0,0,0")

### PICKLES in LSST filters (for DC1, star-gal)

In [ ]:
maglib.run(typ="S",
    star_lib_in="PICKLES_SEDs",
    star_lib_out="PICKLES_LSST",
    filter_file="photozdc1",
    magtype="AB",
    mod_extinc="0",
    extinc_law="calzetti.dat",
    em_dispersion="0",
    lib_ascii="YES",
    z_step="0.0,0,0")

### BT in DES filters

In [ ]:
maglib.run(typ="S",
    star_lib_in="BT_SEDs",
    star_lib_out="BT_DES",
    filter_file="photozDES",
    magtype="AB",
    mod_extinc="0",
    extinc_law="calzetti.dat",
    em_dispersion="0",
    lib_ascii="YES",
    z_step="0.0,0,0")

### BT in LSST filters

In [ ]:
maglib.run(typ="S",
    star_lib_in="BT_SEDs",
    star_lib_out="BT_LSST",
    filter_file="photozdc1",
    magtype="AB",
    mod_extinc="0",
    extinc_law="calzetti.dat",
    em_dispersion="0",
    lib_ascii="YES",
    z_step="0.0,0,0")

## Run photoz

Finally run the photoz. We try it for two different photometry catalogs: calibrated star from DES catalog (in DES filters) and DC1 simulation catalog (in LSST filters).

### DC1


Initialization

In [ ]:
config.update(
    {
        "ZPHOTLIB": "buzzard_LSST,PICKLES_LSST",
        "SPEC_OUT": "NO",
        "CAT_IN": f"{base_dir}/stargal/catalogs/DC1/Shuffle_Buzzard_training_file.dat",
        "INP_TYPE": "M",
        "CAT_TYPE": "LONG",
        "CAT_MAG": "AB",
        "PARA_OUT": f"{base_dir}/stargal/catalogs/star_gal/stargal_output.para",
        "CAT_OUT": f"{base_dir}/stargal/catalogs/DC1/DC1_Buzzard_PICKLES_LSST.out",
        "ZPHOTLIB": "buzzard_LSST,PICKLES_LSST",
        "AUTO_ADAPT": "NO",
        "STAR_PDF_OUT": f"{base_dir}/stargal/catalogs/DC1/PDFs/DC1_Buzzard_PICKLES_LSST_kk",
        "PDZ_OUT": f"{base_dir}/stargal/catalogs/DC1/PDZs/DC1_Buzzard_PICKLES_LSST_piton",
        "PDZ_TYPE": "MIN_ZG",
        "CAT_FMT": "MMEE",
        "ERR_SCALE": 0,
        "SPEC_OUT": "NO",
        "CHI2_OUT": "NO"
    }
)

zphota = lp.PhotoZ(lp.all_types_to_keymap(config))

Load catalog

In [ ]:
cat = np.loadtxt(f"{base_dir}/stargal/catalogs/DC1/Shuffle_Buzzard_training_file.dat")
id = cat[:, 0]
mag = cat[:, 1:7]
emag = cat[:, 7:13]
context = cat[:, 13]
zspec = cat[:, 14]
print("Check format with context and zspec :", context, zspec)

Turn rows into lephare onsources

In [ ]:
srclist = []
n = 5
for i in range(n):
    oneObj = lp.onesource(i, zphota.gridz)
    oneObj.readsource(str(id[i]), mag[i, :], emag[i, :], int(context[i]), zspec[i], " ")
    zphota.prep_data(oneObj)
    srclist.append(oneObj)
print("Sources with a spec-z: ", len(srclist))

In [ ]:
zphota.run_photoz(srclist[:n], [])

# photz.run_photoz(photozlist, a0) #if autao adapt on

## Turn output in fits

In [ ]:
t = zphota.build_output_tables(srclist[:n], para_out=f"{base_dir}/training_stats/simulation_catalogs/star_gal/stargal_output.para", filename="DESstars_Buzzard_PICKLES_DES.fits")

In [ ]:
t[:5]

In [ ]:
import time

zphota.write_outputs(srclist[:n], int(time.time()))

In [ ]:
# This created the output ascii file specified in the config CAT_OUT parameter
!ls -al zphota.out